# Hazeon Hindi Translator

Translate English DOCX files to formal Hindi (UPSC/HCS-style) using **OpenAI GPT-4o** (primary) + **Gemini 2.5 Flash** (fallback). Nirmala UI font applied. Multi-pass repair for 97–100% accuracy.

## One-time setup (first time only)

Get TWO API keys (both needed so the fallback can kick in when one provider has an outage):

1. **OpenAI key** at https://platform.openai.com/api-keys (primary — pay-as-you-go, add $5-10 credit to start)
2. **Gemini key** at https://aistudio.google.com/apikey (fallback — free tier is fine)
3. Open this notebook, click **File → Save a copy in Drive** (so you have your own private version)
4. Click the **🔑 key icon** in the left sidebar of Colab, add both secrets:
   - Name `OPENAI_API_KEY`, Value: your OpenAI key, toggle **Notebook access** ON
   - Name `GEMINI_API_KEY`, Value: your Gemini key, toggle **Notebook access** ON
5. Bookmark your saved notebook URL

## Daily usage

1. Run **Cell 1 (Setup)** — takes ~15 sec after first day, ~3 min first day
2. Run **Cell 2 (Translate)** — pick subject, upload DOCX, wait for download
3. Done!

**Do not paste your API keys into any code cell** — Colab Secrets is the safe place.


In [ ]:
#@title 🔧 Setup (run once per session) { display-mode: "form" }
import os, subprocess, sys
from google.colab import drive, userdata

REPO_URL = 'https://github.com/HR-DHULL/hazeon-hindi-translator.git'
REPO_TAG = 'colab-v1'   # pinned version — updated explicitly, never silently
CACHE_DIR = '/content/drive/MyDrive/hazeon_cache'
REPO_DIR = '/content/hazeon'

# Verify Node is present on this Colab runtime
try:
  node_ver = subprocess.check_output(['node', '--version']).decode().strip()
  print(f'✓ Node {node_ver}')
except Exception:
  print('Installing Node.js (one-time, ~90 sec)...')
  subprocess.run(['apt-get', 'install', '-y', '-q', 'nodejs', 'npm'], check=True,
                 stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
  node_ver = subprocess.check_output(['node', '--version']).decode().strip()
  print(f'✓ Node {node_ver} installed')

# Mount Drive for node_modules cache (makes daily use viable)
drive.mount('/content/drive')
os.makedirs(CACHE_DIR, exist_ok=True)

# Clone (or refresh) the repo at the pinned tag
if os.path.exists(REPO_DIR):
  subprocess.run(['rm', '-rf', REPO_DIR])
subprocess.run(['git', 'clone', '--depth=1', '--branch', REPO_TAG, REPO_URL, REPO_DIR], check=True)
os.chdir(REPO_DIR)

# Restore or populate node_modules cache
cached_modules = f'{CACHE_DIR}/node_modules'
if os.path.exists(cached_modules):
  subprocess.run(['ln', '-sf', cached_modules, f'{REPO_DIR}/node_modules'], check=True)
  print('✓ Using cached node_modules from Drive (15 sec startup)')
else:
  print('First-time install (~3 min) — subsequent sessions will use the Drive cache')
  subprocess.run(['npm', 'install', '--omit=dev', '--no-audit', '--no-fund'], check=True)
  subprocess.run(['cp', '-r', f'{REPO_DIR}/node_modules', cached_modules], check=True)
  print('✓ Cached node_modules to Drive for next time')

# Load both API keys from Colab Secrets → write to .env
env_lines = []
openai_ok = False
gemini_ok = False
try:
  k = userdata.get('OPENAI_API_KEY')
  if k:
    env_lines.append(f'OPENAI_API_KEY={k}')
    env_lines.append('OPENAI_PRIMARY_MODEL=gpt-4o')
    openai_ok = True
    print('✓ OpenAI key loaded from Colab Secrets (primary engine)')
except Exception:
  pass
try:
  k = userdata.get('GEMINI_API_KEY')
  if k:
    env_lines.append(f'GEMINI_API_KEY={k}')
    gemini_ok = True
    print('✓ Gemini key loaded from Colab Secrets (fallback engine)')
except Exception:
  pass

if not openai_ok and not gemini_ok:
  print('')
  print('✗ No API keys found. Add at least OPENAI_API_KEY in Colab Secrets (🔑 icon).')
  print('  Name = OPENAI_API_KEY, Value = your key, toggle Notebook access ON')
  sys.exit(1)

if not openai_ok:
  print('⚠ OPENAI_API_KEY missing — translation will rely on Gemini only (slower, less reliable).')
if not gemini_ok:
  print('⚠ GEMINI_API_KEY missing — no fallback if OpenAI fails.')

with open(f'{REPO_DIR}/.env', 'w') as f:
  f.write('\n'.join(env_lines) + '\n')

# Health check summary
commit = subprocess.check_output(['git', 'rev-parse', '--short', 'HEAD']).decode().strip()
print(f'')
print(f'✓ Ready — version {REPO_TAG} @ {commit}')
print(f'  Now run Cell 2 to translate a document.')


In [ ]:
#@title 📄 Translate a document { display-mode: "form" }
#@markdown **1.** Pick the subject that best matches your document. `auto-detect` lets the engine decide per paragraph — fine for mixed content.
SUBJECT = "history" #@param ["history", "geography", "economics", "polity", "science", "environment", "auto-detect"]

#@markdown **2.** Click ▶ (Play) on the left, then upload your DOCX when prompted.

import os, subprocess, shutil, time
from google.colab import files

REPO_DIR = '/content/hazeon'
if not os.path.exists(REPO_DIR):
  print('✗ Setup not complete. Run Cell 1 first.')
  raise SystemExit
os.chdir(REPO_DIR)

print('Upload your English DOCX file below...')
uploaded = files.upload()
if not uploaded:
  raise SystemExit('No file uploaded — re-run this cell to try again.')

input_name = next(iter(uploaded.keys()))
ts = time.strftime('%H%M%S')
safe_name = input_name.replace(' ', '_')
input_path = f'/content/in_{ts}_{safe_name}'
shutil.move(input_name, input_path)

print(f'\n📄 Translating {input_name} (subject: {SUBJECT})')
print('   Progress streams below — this takes 2–15 min depending on size.')
print('=' * 70)

cmd = ['node', 'translate_local.js']
if SUBJECT != 'auto-detect':
  cmd.append(f'--subject={SUBJECT}')
cmd.append(input_path)

result = subprocess.run(cmd)
print('=' * 70)

base = os.path.splitext(os.path.basename(input_path))[0]
output_path = f'/content/{base}_hindi.docx'

if result.returncode == 0 and os.path.exists(output_path):
  drive_out_dir = '/content/drive/MyDrive/hazeon_outputs'
  os.makedirs(drive_out_dir, exist_ok=True)
  drive_out = f'{drive_out_dir}/{base}_hindi.docx'
  shutil.copy(output_path, drive_out)
  print(f'\n✓ Also saved to Drive: {drive_out}')
  print('  (if the browser download below fails, open Google Drive to get the file)')
  print('\n⬇ Browser download starting...')
  files.download(output_path)
else:
  print('\n✗ Translation failed — scroll up for the error message.')


In [ ]:
#@title 📊 Health check (optional diagnostic) { display-mode: "form" }
#@markdown Run this if Cell 2 is failing — tests both API keys.
import os, subprocess
if not os.path.exists('/content/hazeon'):
  print('✗ Run Cell 1 (Setup) first.')
else:
  os.chdir('/content/hazeon')
  subprocess.run(['node', '-e', '''
import("dotenv/config").then(async () => {
  // Test OpenAI
  if (process.env.OPENAI_API_KEY) {
    const OpenAI = (await import("openai")).default;
    const c = new OpenAI({ apiKey: process.env.OPENAI_API_KEY });
    const t = Date.now();
    try {
      const r = await c.chat.completions.create({
        model: "gpt-4o", max_tokens: 50,
        messages: [{role: "user", content: "Translate to Hindi: The Ganges is a sacred river."}],
      });
      console.log(`✓ OpenAI works (${Date.now()-t}ms) — ${r.choices[0].message.content.slice(0,80)}`);
    } catch (e) {
      console.log("✗ OpenAI failed:", e.message.slice(0, 200));
    }
  } else console.log("✗ No OPENAI_API_KEY in .env");

  // Test Gemini
  if (process.env.GEMINI_API_KEY) {
    const { GoogleGenerativeAI } = await import("@google/generative-ai");
    const g = new GoogleGenerativeAI(process.env.GEMINI_API_KEY);
    const m = g.getGenerativeModel({ model: "gemini-2.5-flash" });
    const t = Date.now();
    try {
      const r = await m.generateContent("Translate to Hindi: The Ganges is a sacred river.");
      console.log(`✓ Gemini works (${Date.now()-t}ms) — ${r.response.text().slice(0,80)}`);
    } catch (e) {
      console.log("✗ Gemini failed:", e.message.slice(0, 200));
    }
  } else console.log("✗ No GEMINI_API_KEY in .env");
});
'''])


## Troubleshooting

| Problem | Fix |
|---|---|
| `OPENAI_API_KEY not found` | Add it in Colab Secrets (🔑 icon, left sidebar). Toggle Notebook access ON. |
| `You exceeded your current quota` | Your OpenAI balance ran out. Add funds at https://platform.openai.com/settings/organization/billing/overview |
| OpenAI fails, Gemini works | Check your OpenAI balance. The fallback handled it but primary is better. |
| Cell 1 fails at `git clone` | GitHub is temporarily down — wait a minute, re-run. |
| Cell 1 fails at `npm install` | Delete the cache folder at `MyDrive/hazeon_cache` in Google Drive, re-run Cell 1. |
| Cell 2 says `429` or `rate-limit` | You hit an API quota. OpenAI: add credits. Gemini free: wait a minute. |
| Download didn't start (Safari etc.) | Open Google Drive → `hazeon_outputs/` folder → download from there. |
| Output file has English paragraphs | Run Cell 2 again on the same file — each pass fixes more. 2-3 passes usually enough. |
| Long docs timing out | Colab free-tier sessions cap at 12 hours. For very large docs, use the desktop CLI. |

**Support:** message Harsh in the team chat with the error message + your filename.
